# CHUYÊN ĐỀ SỐNG CÒN CỦA DATA ENGINEER: XỬ LÝ BIẾN ĐỘNG SCHEMA BẤT NGỜ (SCHEMA DRIFT RESILIENCE)
## KIẾN TRÚC ĐA LAKEHOUSE VỚI CÚ PHÁP 3 THÀNH PHẦN: `ten_lakehouse.dbo.ten_bang` TRÊN FABRIC

Trên Microsoft Fabric với cơ chế Lakehouse Schemas, các bảng được định danh chuẩn xác qua cú pháp 3 thành phần:
- **`lh_bronze.dbo.bronze_customers_resilient`**: Bảng raw ingest từ Kafka/MinIO trên Lakehouse `lh_bronze`.
- **`lh_silver.dbo.silver_customers_resilient`**: Bảng dữ liệu sạch chuẩn hóa trên Lakehouse `lh_silver`.
- **`lh_silver.dbo.quarantine_customers`**: Bảng cách ly lỗi (Dead-Letter Queue) trên Lakehouse `lh_silver`.
- **`wh_gold`**: Synapse Data Warehouse phục vụ báo cáo Star Schema (Fact / Dim).

> **LƯU Ý KHI CHẠY TRÊN FABRIC:**
> Tại thanh **Explorer** (cột bên trái Notebook), hãy đảm bảo đã bấm **+ Lakehouse** và thêm cả 2 lakehouse **`lh_bronze`** và **`lh_silver`**.

## BƯỚC 1: KHỞI TẠO MẺ DỮ LIỆU BAN ĐẦU (VERSION 1)
- Ghi dữ liệu thô dạng JSON string vào bảng **`lh_bronze.dbo.bronze_customers_resilient`**.
- Parse JSON và ghi sang bảng **`lh_silver.dbo.silver_customers_resilient`**.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import datetime

# Định nghĩa tên Lakehouse và Schema chuẩn Fabric (ten_lakehouse.dbo.ten_bang)
LH_BRONZE = "lh_bronze.dbo"
LH_SILVER = "lh_silver.dbo"

# 1. Giả lập tin nhắn từ Kafka vào Bronze Lakehouse (Lưu nguyên raw payload JSON string)
v1_kafka_raw = [
    ("k_offset_01", datetime(2026, 1, 1, 8, 0, 0), '{"customer_id": "CUST_101", "customer_name": "Nguyen Van An", "city": "Hanoi"}'),
    ("k_offset_02", datetime(2026, 1, 1, 8, 1, 0), '{"customer_id": "CUST_102", "customer_name": "Tran Thi Binh", "city": "Danang"}')
]
bronze_schema = StructType([
    StructField("kafka_offset", StringType(), False),
    StructField("ingest_timestamp", TimestampType(), False),
    StructField("raw_payload", StringType(), False) # Giữ nguyên String để không bao giờ bị sập
])

df_bronze_v1 = spark.createDataFrame(v1_kafka_raw, bronze_schema)
# Ghi trực tiếp vào lh_bronze.dbo.bronze_customers_resilient
df_bronze_v1.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{LH_BRONZE}.bronze_customers_resilient")

print(f"=== BẢNG BRONZE V1 TRONG {LH_BRONZE}.bronze_customers_resilient ===")
spark.table(f"{LH_BRONZE}.bronze_customers_resilient").show(truncate=False)

# 2. Xử lý tầng Silver V1: Đọc từ lh_bronze.dbo, Parse JSON và ghi sang lh_silver.dbo
v1_json_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True)
])

df_silver_v1 = spark.table(f"{LH_BRONZE}.bronze_customers_resilient") \
    .withColumn("parsed", F.from_json(F.col("raw_payload"), v1_json_schema)) \
    .select("parsed.*", "ingest_timestamp")

# Ghi trực tiếp sang lh_silver.dbo.silver_customers_resilient
df_silver_v1.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{LH_SILVER}.silver_customers_resilient")

print(f"=== BẢNG SILVER V1 TRONG {LH_SILVER}.silver_customers_resilient ===")
spark.table(f"{LH_SILVER}.silver_customers_resilient").show(truncate=False)


## BƯỚC 2: SỰ CỐ ĐỘT NGỘT - NGUỒN ORACLE THAY ĐỔI SCHEMA BẤT NGỜ (VERSION 2)

Bên nguồn Oracle tự ý thay đổi ứng dụng mà **KHÔNG BÁO TRƯỚC**:
1. **Thêm 2 cột mới:** `tax_code` (Mã số thuế) và `loyalty_points` (Điểm tích lũy).
2. **Không gửi cột cũ:** Cột `city` không còn xuất hiện trong JSON payload.
3. **Lỗi cú pháp:** Kèm 1 tin nhắn JSON bị lỗi gãy cú pháp do ký tự đặc biệt.

👉 Dữ liệu vẫn được đổ an toàn vào **`lh_bronze.dbo.bronze_customers_resilient`** vì tầng Bronze chỉ tiếp nhận chuỗi thô (Raw String).

In [ ]:
v2_kafka_raw = [
    # Bản ghi 1: Khách cũ CUST_101 có thêm 2 cột mới (tax_code, loyalty_points) và không còn cột city
    ("k_offset_03", datetime(2026, 1, 1, 9, 0, 0), '{"customer_id": "CUST_101", "customer_name": "Nguyen Van An", "tax_code": "010998877", "loyalty_points": 250}'),
    # Bản ghi 2: Khách mới CUST_103
    ("k_offset_04", datetime(2026, 1, 1, 9, 5, 0), '{"customer_id": "CUST_103", "customer_name": "Le Van Cuong", "tax_code": "030112233", "loyalty_points": 100}'),
    # Bản ghi 3: TIN NHẮN RÁC BỊ LỖI CÚ PHÁP JSON (Đoạn text bị gãy gập)
    ("k_offset_05", datetime(2026, 1, 1, 9, 10, 0), '{"customer_id": "CUST_999", BAD_JSON_SYNTAX_HERE!!}')
]

df_bronze_v2 = spark.createDataFrame(v2_kafka_raw, bronze_schema)
# Ghi Append tiếp vào lh_bronze.dbo.bronze_customers_resilient: 100% THÀNH CÔNG VÌ CHỈ LƯU RAW STRING!
df_bronze_v2.write.format("delta").mode("append").saveAsTable(f"{LH_BRONZE}.bronze_customers_resilient")

print(f"=== BẢNG {LH_BRONZE}.bronze_customers_resilient ĐÃ TIẾP NHẬN BATCH 2 AN TOÀN ===")
spark.table(f"{LH_BRONZE}.bronze_customers_resilient").filter("kafka_offset >= 'k_offset_03'").show(truncate=False)


## BƯỚC 3: GIẢI PHÁP PHÒNG THỦ - TÁCH DỮ LIỆU LỖI VÀO BẢNG CÁCH LY TRÊN `lh_silver.dbo`

1. Sử dụng chế độ **`mode = "PERMISSIVE"`** kết hợp với **`columnNameOfCorruptRecord = "_corrupt_record"`**.
2. Tách riêng các dòng bị lỗi JSON đẩy vào bảng **`lh_silver.dbo.quarantine_customers`** (Dead Letter Queue) để bắn cảnh báo cho đội Vận hành (Ops).
3. Các dòng hợp lệ tiếp tục chảy vào luồng chính.

In [ ]:
# Schema mở rộng đón nhận các trường mới và cột bắt lỗi _corrupt_record
v2_permissive_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("tax_code", StringType(), True),           # Cột mới từ Oracle
    StructField("loyalty_points", LongType(), True),       # Cột mới từ Oracle
    StructField("_corrupt_record", StringType(), True)     # Cột cứu cánh: Bắt toàn bộ lỗi JSON hỏng
])

# Đọc mẻ mới từ lh_bronze.dbo và parse an toàn
df_parsed_batch = spark.table(f"{LH_BRONZE}.bronze_customers_resilient") \
    .filter("kafka_offset >= 'k_offset_03'") \
    .withColumn("parsed", F.from_json(F.col("raw_payload"), v2_permissive_schema, {"mode": "PERMISSIVE", "columnNameOfCorruptRecord": "_corrupt_record"})) \
    .select("parsed.*", "kafka_offset", "ingest_timestamp")

# 1. TÁCH BẢN GHI HỎNG ĐẨY SANG BẢNG QUARANTINE TRÊN LAKEHOUSE LH_SILVER
df_quarantine = df_parsed_batch.filter(F.col("_corrupt_record").isNotNull()) \
    .select("kafka_offset", "_corrupt_record", "ingest_timestamp")

df_quarantine.write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable(f"{LH_SILVER}.quarantine_customers")

print(f"=== [DEAD LETTER QUEUE] ĐÃ CÁCH LY BẢN GHI LỖI VÀO {LH_SILVER}.quarantine_customers ===")
spark.table(f"{LH_SILVER}.quarantine_customers").show(truncate=False)

# 2. LỌC DỮ LIỆU HỢP LỆ ĐỂ CHUẨN BỊ MERGE VÀO BẢNG CHÍNH CỦA SILVER
df_valid_clean = df_parsed_batch.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")
print("=== DỮ LIỆU HỢP LỆ SẴN SÀNG MERGE VÀO SILVER ===")
df_valid_clean.show(truncate=False)


## BƯỚC 4: TIẾN HÓA SCHEMA TỰ ĐỘNG (DELTA AUTOMERGE) VÀO `lh_silver.dbo`

Bảng `lh_silver.dbo.silver_customers_resilient` ban đầu chỉ có các cột cũ (`customer_id`, `customer_name`, `city`). 
Bằng việc bật **`spark.databricks.delta.schema.autoMerge.enabled = true`**, câu lệnh `MERGE INTO` sẽ **tự động thêm 2 cột mới `tax_code` và `loyalty_points`** vào bảng đích mà không phát sinh lỗi Schema Mismatch!

In [ ]:
# KÍCH HOẠT AUTOMERGE SCHEMA CHO PHÉP DELTA TỰ TIẾN HÓA KHI MERGE INTO
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

tbl_silver = DeltaTable.forName(spark, f"{LH_SILVER}.silver_customers_resilient")

# Thực hiện MERGE INTO liên Lakehouse theo đúng cú pháp ten_lakehouse.dbo.ten_bang:
tbl_silver.alias("tgt").merge(
    df_valid_clean.alias("src"),
    "tgt.customer_id = src.customer_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print(f"=== BẢNG {LH_SILVER}.silver_customers_resilient SAU KHI TỰ ĐỘNG THÊM CỘT MỚI ===")
spark.table(f"{LH_SILVER}.silver_customers_resilient").show(truncate=False)

print(f"=== KIỂM TRA SCHEMA CỦA {LH_SILVER}.silver_customers_resilient: ===")
spark.table(f"{LH_SILVER}.silver_customers_resilient").printSchema()


## BƯỚC 5: KẾT NỐI TẦNG GOLD (FABRIC WAREHOUSE `wh_gold`)

Sau khi dữ liệu đã nằm an toàn trong **`lh_silver.dbo`**, tầng Gold trên **Fabric Synapse Warehouse (`wh_gold`)** có thể truy vấn trực tiếp bảng này qua cơ chế **Cross-Database Query** trong T-SQL của Fabric mà không cần phải copy dữ liệu:

```sql
-- CÂU LỆNH T-SQL CHẠY TRỰC TIẾP TRÊN WAREHOUSE wh_gold (HOẶC QUA dbt-fabric):
SELECT 
    customer_id,
    customer_name,
    COALESCE(city, 'Chưa cập nhật') AS city,
    COALESCE(tax_code, 'Không có') AS tax_code,
    COALESCE(loyalty_points, 0) AS loyalty_points
FROM [lh_silver].[dbo].[silver_customers_resilient];
```

👉 **Toàn vẹn kiến trúc:** `Oracle ➔ Kafka ➔ MinIO ➔ lh_bronze.dbo ➔ lh_silver.dbo ➔ wh_gold` vận hành thông suốt, thích ứng với mọi biến động Schema của nguồn!